[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/01_collection/A9_city_profile_builder.ipynb)

# A9: City Profile Builder

---

## Purpose

This notebook helps you **adapt the Berkeley housing pipeline to your own city**.

It reads a city configuration file (YAML) and generates:
1. A city profile summary
2. A checklist of data sources to set up
3. Status code mappings for your permit system

## Getting Started

1. Copy `city_template.yaml` to `city_<yourCity>.yaml`
2. Fill in your city's information
3. Run this notebook
4. Follow the generated checklist to adapt the A-D notebooks

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
from datetime import datetime

# Try to import yaml (install if needed)
try:
    import yaml
except ImportError:
    print("Installing PyYAML...")
    !pip install pyyaml
    import yaml

# Find project root
def find_project_root():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() or (path / 'city_template.yaml').exists():
            return path
    return current

ROOT = find_project_root()
print(f"Project root: {ROOT}")

## 2. Load City Configuration

In [ ]:
# Find all city configuration files
city_configs = list(ROOT.glob('city_*.yaml'))

print("Available city configurations:")
for i, config in enumerate(city_configs):
    print(f"  [{i}] {config.name}")

if not city_configs:
    print("\nNo city configs found. Creating from template...")
    print("Copy city_template.yaml to city_<yourCity>.yaml and customize it.")

In [ ]:
# Load a city config
# Change this to load your city's config
CONFIG_FILE = ROOT / 'city_template.yaml'  # Change to your city's file

if CONFIG_FILE.exists():
    with open(CONFIG_FILE) as f:
        city_config = yaml.safe_load(f)
    print(f"Loaded configuration for: {city_config.get('city_name', 'Unknown')}")
else:
    print(f"Config file not found: {CONFIG_FILE}")
    city_config = None

## 3. Generate City Profile

In [ ]:
def generate_city_profile(config):
    """
    Generate a formatted city profile from configuration.
    """
    if not config:
        return "No configuration loaded."
    
    profile = f"""
{'='*60}
CITY PROFILE: {config.get('city_name', 'Unknown')}
{'='*60}

Location:
  City:   {config.get('city_name', 'Unknown')}
  County: {config.get('county', 'Unknown')}
  State:  {config.get('state', 'Unknown')}

Data Sources:
  Open Data Portal:  {config.get('open_data_portal_url', 'Not configured')}
  Permit Portal:     {config.get('permit_portal_url', 'Not configured')}
  GIS/Parcels:       {config.get('gis_parcels_url', 'Not configured')}
  Assessor:          {config.get('assessor_url', 'Not configured')}

Technical Setup:
  Permit System:     {config.get('permit_system_type', 'Unknown')}
  API Available:     {config.get('permit_api_available', False)}
  Geocoding:         {config.get('geocoding_strategy', 'Not configured')}

RHNA Allocation ({config.get('rhna_cycle', 'Unknown')} Cycle, {config.get('rhna_period', 'Unknown')}):
"""
    
    rhna = config.get('rhna_by_income', {})
    profile += f"  Very Low Income:   {rhna.get('very_low', 0):>6} units\n"
    profile += f"  Low Income:        {rhna.get('low', 0):>6} units\n"
    profile += f"  Moderate Income:   {rhna.get('moderate', 0):>6} units\n"
    profile += f"  Above Moderate:    {rhna.get('above_moderate', 0):>6} units\n"
    profile += f"  {'─'*25}\n"
    profile += f"  TOTAL:             {config.get('rhna_total_units', 0):>6} units\n"
    
    profile += f"\nMaintainer: {config.get('maintainer', 'Not specified')}\n"
    profile += f"Last Updated: {config.get('last_updated', 'Unknown')}\n"
    
    return profile

# Display profile
print(generate_city_profile(city_config))

## 4. Generate Setup Checklist

In [ ]:
def generate_checklist(config):
    """
    Generate a checklist for setting up the pipeline for a new city.
    """
    if not config:
        return "No configuration loaded."
    
    city = config.get('city_name', 'Your City')
    
    checklist = f"""
{'='*60}
SETUP CHECKLIST: {city}
{'='*60}

Phase 1: Data Collection (A-Series)
─────────────────────────────────────
[ ] A1: Configure data sources
    • Test Open Data Portal API: {config.get('open_data_portal_url', 'N/A')}
    • Verify permit data access: {config.get('permit_portal_url', 'N/A')}
    • Check for API keys/authentication needed

[ ] A2: Address standardization
    • Review address formats in your city
    • Add any city-specific abbreviations
    • Test with 10 sample addresses

[ ] A3: Geocoding setup
    • Strategy: {config.get('geocoding_strategy', 'Not configured')}
    • Obtain county address database (if using county_address_db)
    • Or configure Census/Google/Mapbox API

[ ] A4: APN enrichment
    • Get county assessor parcel data
    • Verify APN format for {config.get('county', 'your county')}
    • Test address-to-APN matching

Phase 2: Timeline Tracking (B-Series)
─────────────────────────────────────
[ ] B2: Status classification
    • Map your permit statuses (see config: status_mappings)
    • Test with sample permits

Phase 3: Analysis (C-Series)
─────────────────────────────────────
[ ] C1-C3: Analysis notebooks
    • Update RHNA targets: {config.get('rhna_total_units', 0)} units
    • Adjust date ranges for your data

Phase 4: Reporting (D-Series)
─────────────────────────────────────
[ ] D4: HCD APR tables
    • Verify APR field mappings for your data
    • Test with one year of data

Phase 5: Deployment
─────────────────────────────────────
[ ] Create city-specific database
[ ] Deploy to Datasette (or similar)
[ ] Update documentation with city name

Notes:
{config.get('notes', 'No additional notes.')}
"""
    return checklist

print(generate_checklist(city_config))

## 5. Status Code Mapping

In [ ]:
def display_status_mappings(config):
    """
    Display configured status code mappings.
    """
    if not config or 'status_mappings' not in config:
        print("No status mappings configured.")
        return
    
    print("\n" + "="*60)
    print("STATUS CODE MAPPINGS")
    print("="*60)
    print("\nYour permit statuses → Standard categories:\n")
    
    for category, statuses in config['status_mappings'].items():
        print(f"  {category.upper()}:")
        for status in statuses:
            print(f"    • {status}")
        print()

display_status_mappings(city_config)

## 6. Save City Profile

In [ ]:
def save_city_profile(config, output_dir):
    """
    Save city profile as a markdown file.
    """
    if not config:
        print("No configuration to save.")
        return
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    city_name = config.get('city_name', 'unknown').lower().replace(' ', '_')
    output_path = output_dir / f'city_profile_{city_name}.md'
    
    content = f"""# City Profile: {config.get('city_name', 'Unknown')}

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Overview

| Field | Value |
|-------|-------|
| City | {config.get('city_name', 'Unknown')} |
| County | {config.get('county', 'Unknown')} |
| State | {config.get('state', 'Unknown')} |
| RHNA Cycle | {config.get('rhna_cycle', 'Unknown')} |
| RHNA Total | {config.get('rhna_total_units', 0):,} units |

## Data Sources

| Source | URL |
|--------|-----|
| Open Data Portal | {config.get('open_data_portal_url', 'Not configured')} |
| Permit Portal | {config.get('permit_portal_url', 'Not configured')} |
| GIS/Parcels | {config.get('gis_parcels_url', 'Not configured')} |
| Assessor | {config.get('assessor_url', 'Not configured')} |

## Technical Configuration

- **Permit System:** {config.get('permit_system_type', 'Unknown')}
- **API Available:** {config.get('permit_api_available', False)}
- **Geocoding Strategy:** {config.get('geocoding_strategy', 'Not configured')}

## RHNA Allocation

| Income Category | Units |
|-----------------|-------|
| Very Low Income | {config.get('rhna_by_income', {}).get('very_low', 0):,} |
| Low Income | {config.get('rhna_by_income', {}).get('low', 0):,} |
| Moderate Income | {config.get('rhna_by_income', {}).get('moderate', 0):,} |
| Above Moderate | {config.get('rhna_by_income', {}).get('above_moderate', 0):,} |
| **Total** | **{config.get('rhna_total_units', 0):,}** |

## Notes

{config.get('notes', 'No additional notes.')}
"""
    
    with open(output_path, 'w') as f:
        f.write(content)
    
    print(f"City profile saved to: {output_path}")
    return output_path

# Save the profile
if city_config:
    OUTPUT_DIR = ROOT / 'data/outputs'
    save_city_profile(city_config, OUTPUT_DIR)

---

## Summary

This notebook helps you:

1. **Load a city configuration** from YAML
2. **Generate a city profile** summarizing key information
3. **Create a setup checklist** for adapting the pipeline
4. **Review status code mappings** for your permit system
5. **Save documentation** for your city

## Next Steps

1. Create your city's configuration: `city_<yourCity>.yaml`
2. Run this notebook to generate your profile and checklist
3. Work through the checklist, adapting A-D notebooks
4. Start with `A1_data_sources_setup.ipynb` to test data access

## Contributing

If you adapt this pipeline to a new city, consider:
- Sharing your city configuration (PR welcome!)
- Documenting any unique challenges
- Contributing improvements to the core notebooks

**Goal:** Help every California city track and understand its housing pipeline.